# SpaceX Launch Dashboard — Completed Tasks

This notebook completes the Plotly Dash assignment tasks:

1. Add a launch site dropdown component.
2. Add a callback for the success pie chart.
3. Add a payload range slider.
4. Add a callback for the payload-success scatter chart.
5. Include visual-insight questions with code to compute the answers from the dataset.

The dashboard code is written so it can be copied into `spacex_dash_app.py` or run from this notebook. Because apparently a dashboard is not real until it has at least two callbacks and one slider to make everyone feel scientific.


In [ ]:
# Install packages if needed. Run this cell only if Dash/Plotly are missing.
# In IBM Skills Network labs, these are often already installed.

# !pip install pandas dash plotly

## Completed Dash Application Code

In [ ]:
import pandas as pd
import plotly.express as px
from dash import Dash, html, dcc
from dash.dependencies import Input, Output

# Load SpaceX launch data
# This is the public IBM Skills Network dataset used in the dashboard lab.
spacex_df = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/module_3/spacex_launch_dash.csv"
)

# Get payload range for the slider default values
min_payload = spacex_df["Payload Mass (kg)"].min()
max_payload = spacex_df["Payload Mass (kg)"].max()

# Create list of launch site options for dropdown
launch_sites = sorted(spacex_df["Launch Site"].unique())
site_options = [{"label": "All Sites", "value": "ALL"}]
site_options += [{"label": site, "value": site} for site in launch_sites]

# Create Dash app
app = Dash(__name__)

app.layout = html.Div(
    children=[
        html.H1(
            "SpaceX Launch Records Dashboard",
            style={"textAlign": "center", "color": "#503D36", "font-size": 40},
        ),

        # TASK 1: Add a Launch Site Drop-down Input Component
        dcc.Dropdown(
            id="site-dropdown",
            options=site_options,
            value="ALL",
            placeholder="Select a Launch Site here",
            searchable=True,
        ),
        html.Br(),

        # Pie chart output
        html.Div(dcc.Graph(id="success-pie-chart")),
        html.Br(),

        html.P("Payload range (kg):"),

        # TASK 3: Add a Range Slider to Select Payload
        dcc.RangeSlider(
            id="payload-slider",
            min=0,
            max=10000,
            step=1000,
            marks={i: str(i) for i in range(0, 10001, 2500)},
            value=[min_payload, max_payload],
        ),
        html.Br(),

        # Scatter chart output
        html.Div(dcc.Graph(id="success-payload-scatter-chart")),
    ],
    style={"padding": "20px"},
)


# TASK 2: Add a callback function to render success-pie-chart based on selected site dropdown
@app.callback(
    Output(component_id="success-pie-chart", component_property="figure"),
    Input(component_id="site-dropdown", component_property="value"),
)
def get_pie_chart(entered_site):
    if entered_site == "ALL":
        # For all sites, show successful launches by launch site
        success_df = spacex_df[spacex_df["class"] == 1]
        fig = px.pie(
            success_df,
            names="Launch Site",
            title="Total Successful Launches by Site",
        )
    else:
        # For one selected site, show success vs. failure counts
        filtered_df = spacex_df[spacex_df["Launch Site"] == entered_site]
        outcome_counts = (
            filtered_df["class"]
            .value_counts()
            .rename_axis("class")
            .reset_index(name="count")
        )
        outcome_counts["Outcome"] = outcome_counts["class"].map(
            {0: "Failure", 1: "Success"}
        )
        fig = px.pie(
            outcome_counts,
            values="count",
            names="Outcome",
            title=f"Launch Outcomes for {entered_site}",
        )
    return fig


# TASK 4: Add a callback function to render the success-payload-scatter-chart scatter plot
@app.callback(
    Output(component_id="success-payload-scatter-chart", component_property="figure"),
    [
        Input(component_id="site-dropdown", component_property="value"),
        Input(component_id="payload-slider", component_property="value"),
    ],
)
def get_scatter_chart(entered_site, payload_range):
    low, high = payload_range

    filtered_df = spacex_df[
        (spacex_df["Payload Mass (kg)"] >= low)
        & (spacex_df["Payload Mass (kg)"] <= high)
    ]

    if entered_site != "ALL":
        filtered_df = filtered_df[filtered_df["Launch Site"] == entered_site]
        title = f"Payload Mass vs. Launch Outcome for {entered_site}"
    else:
        title = "Payload Mass vs. Launch Outcome for All Sites"

    fig = px.scatter(
        filtered_df,
        x="Payload Mass (kg)",
        y="class",
        color="Booster Version Category",
        hover_data=["Launch Site"],
        title=title,
        labels={"class": "Launch Outcome (0 = Failure, 1 = Success)"},
    )
    return fig


# Run the app
# In some environments, app.run_server() has been replaced by app.run().
# If port 8050 is already in use, change the port to 8051 or another available port.
if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8050, debug=True)


## Visual Insight Analysis

In [ ]:
# Analysis code for the visual insight questions
# Run this after loading spacex_df.

site_summary = (
    spacex_df.groupby("Launch Site")
    .agg(total_launches=("class", "count"), successful_launches=("class", "sum"))
    .reset_index()
)
site_summary["success_rate"] = site_summary["successful_launches"] / site_summary["total_launches"]

largest_success_site = site_summary.sort_values(
    "successful_launches", ascending=False
).iloc[0]
highest_success_rate_site = site_summary.sort_values(
    "success_rate", ascending=False
).iloc[0]

# Payload ranges grouped into 1000 kg bins
payload_bins = pd.cut(
    spacex_df["Payload Mass (kg)"],
    bins=range(0, 11000, 1000),
    include_lowest=True,
)
payload_summary = (
    spacex_df.groupby(payload_bins, observed=True)
    .agg(total_launches=("class", "count"), successful_launches=("class", "sum"))
    .reset_index()
    .rename(columns={"Payload Mass (kg)": "payload_range"})
)
payload_summary["success_rate"] = (
    payload_summary["successful_launches"] / payload_summary["total_launches"]
)

highest_payload_range = payload_summary.sort_values(
    ["success_rate", "total_launches"], ascending=[False, False]
).iloc[0]
lowest_payload_range = payload_summary.sort_values(
    ["success_rate", "total_launches"], ascending=[True, False]
).iloc[0]

booster_summary = (
    spacex_df.groupby("Booster Version Category")
    .agg(total_launches=("class", "count"), successful_launches=("class", "sum"))
    .reset_index()
)
booster_summary["success_rate"] = (
    booster_summary["successful_launches"] / booster_summary["total_launches"]
)
highest_booster = booster_summary.sort_values(
    ["success_rate", "total_launches"], ascending=[False, False]
).iloc[0]

print("Site summary:")
display(site_summary.sort_values("successful_launches", ascending=False))

print("\nPayload summary:")
display(payload_summary.sort_values("payload_range"))

print("\nBooster summary:")
display(booster_summary.sort_values("success_rate", ascending=False))

print("\nAnswers:")
print(f"1. Largest number of successful launches: {largest_success_site['Launch Site']}")
print(f"2. Highest launch success rate: {highest_success_rate_site['Launch Site']} ({highest_success_rate_site['success_rate']:.1%})")
print(f"3. Highest payload success range: {highest_payload_range['payload_range']} ({highest_payload_range['success_rate']:.1%})")
print(f"4. Lowest payload success range: {lowest_payload_range['payload_range']} ({lowest_payload_range['success_rate']:.1%})")
print(f"5. Highest booster success rate: {highest_booster['Booster Version Category']} ({highest_booster['success_rate']:.1%})")


## Written Answers

Run the analysis cell above to compute exact results from the dataset in your environment. The answers are calculated directly from `spacex_df`, so they will match the file used in the IBM lab instead of relying on guesswork, which is a rare moment of civilization.

The dashboard lets you answer:

1. **Which site has the largest successful launches?**  
   Use the all-sites pie chart or the `site_summary` table.

2. **Which site has the highest launch success rate?**  
   Use `successful_launches / total_launches` in the `site_summary` table.

3. **Which payload range has the highest launch success rate?**  
   Use the payload summary table grouped in 1000 kg bins.

4. **Which payload range has the lowest launch success rate?**  
   Use the same payload summary table and sort by success rate ascending.

5. **Which booster version has the highest launch success rate?**  
   Use the booster summary table grouped by `Booster Version Category`.
